# 03d — CP2 v2 production migration / refreeze audit

Purpose: promote the already-audited Notebook-03 timezone contract into production and verify it on the frozen 5,821 CP1 stays.

This notebook does **not** re-research timezone semantics. It checks that production now matches the reviewed candidate:

`(lat, lon) → IANA timezone_id → ZoneInfo → local wall clock → complete-link 200 m → HOME/OFFICE evidence`.

The old 27 HOME / 16 OFFICE result is a historical CP2-v1 comparator, not a parity target.


In [ ]:
from pathlib import Path
from hashlib import sha256
import importlib.util, json, os, subprocess, sys

REPO_URL='https://github.com/tanh1c/geolife.git'
BRANCH='feat/cp2-v2-local-timezone-production'
REPO_DIR=Path('/root/geolife')

if not (REPO_DIR/'.git').exists():
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO_DIR)],check=True)
else:
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'checkout',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard',f'origin/{BRANCH}'],check=True)

subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[dev]'],cwd=REPO_DIR,check=True)
for p in (str(REPO_DIR),str(REPO_DIR/'src')):
    if p not in sys.path: sys.path.insert(0,p)
GIT_SHA=subprocess.run(['git','-C',str(REPO_DIR),'rev-parse','HEAD'],capture_output=True,text=True,check=True).stdout.strip()
print('sha:',GIT_SHA[:12])


In [ ]:
import pandas as pd
from fastapi.testclient import TestClient

spec=importlib.util.spec_from_file_location('stage03d',REPO_DIR/'analysis'/'03d_cp2_v2_production_migration.py')
stage03d=importlib.util.module_from_spec(spec); sys.modules[spec.name]=stage03d; spec.loader.exec_module(stage03d)
print(stage03d.synthetic_self_check())

from geolife.api.app import app
from geolife.model import build_semantic_locations, infer_home_office

V=Path('/mnt/geolife-data')
if not V.exists(): raise FileNotFoundError('Attach the GeoLife Modal Volume at /mnt/geolife-data')
C=V/'cache'/'03d_cp2_v2_production_migration'; C.mkdir(parents=True,exist_ok=True)

def find_stay_cache():
    candidates=[
        V/'cache'/'cp2_home_office'/'stays_baseline_v1.pkl',
        V/'cache'/'03a_user_behavior_deep_dive'/'stays_baseline_v1.pkl',
        V/'artifacts'/'03a'/'stays_baseline_v1.pkl',
    ]
    for path in candidates:
        if path.exists(): return path
    hits=sorted(V.glob('**/stays_baseline_v1.pkl'))
    if hits: return hits[0]
    raise FileNotFoundError('stays_baseline_v1.pkl')

STAY_CACHE=find_stay_cache()
stays=pd.read_pickle(STAY_CACHE)
assert len(stays)==5821
assert stays['user_id'].nunique()==136
print('stay cache:',STAY_CACHE)
print('stays:',len(stays),'users:',stays['user_id'].nunique())

def file_sha256(path):
    h=sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b''): h.update(chunk)
    return h.hexdigest()

STAY_CACHE_SHA256=file_sha256(STAY_CACHE)
print('stay cache sha256:',STAY_CACHE_SHA256)


## 1. Independent Notebook-03 reference vs production v2

Reference timezone/clustering/scoring is reconstructed independently in `analysis/03d_cp2_v2_production_migration.py`. Production raw cluster IDs are not compared directly; cluster signatures and emitted evidence are compared instead.


In [ ]:
comparison=stage03d.run_migration_comparison(stays)
print('MIGRATION DECISION')
display(pd.DataFrame([comparison['decision']]).T.rename(columns={0:'status'}))

print('\nTIMEZONE COVERAGE')
display(pd.Series({
    'cp1_stays':comparison['cp1_stays'],
    'cp1_stay_users':comparison['cp1_stay_users'],
    'timezone_resolved_stays':comparison['timezone_resolved_stays'],
    'timezone_resolved_users':comparison['timezone_resolved_users'],
    'distinct_timezones':comparison['distinct_timezones'],
    'location_namespace':comparison['location_namespace'],
}).to_frame('value'))

print('\nTOP TIMEZONES')
display(pd.Series(comparison['timezone_counts']).sort_values(ascending=False).head(20).to_frame('stays'))

print('\nLOCAL-TIME PARITY')
display(pd.Series(comparison['local_time_parity']).to_frame('value'))
print('\nLOCATION PARITY')
display(pd.Series(comparison['location_parity']).to_frame('value'))
print('\nEMISSION PARITY')
display(pd.Series(comparison['emission_parity']).to_frame('value'))

historical=comparison['historical_v1']
current={
    'semantic_users':comparison['timezone_resolved_users'],
    'semantic_locations':comparison['location_parity']['production_locations'],
    'recurring_locations':comparison['location_parity']['production_recurring_locations'],
    'recurring_users':comparison['location_parity']['production_recurring_users'],
    'HOME':comparison['emission_parity']['production_counts']['HOME'],
    'OFFICE':comparison['emission_parity']['production_counts']['OFFICE'],
}
delta=pd.DataFrame({'historical_cp2_v1':historical,'production_cp2_v2':current})
delta['delta_v2_minus_v1']=delta['production_cp2_v2']-delta['historical_cp2_v1']
print('\nV1 → V2 MEASURED DELTA')
display(delta)

if comparison['decision']['runner_status']!='ready_for_cp2_v2_review':
    raise RuntimeError(comparison['decision'])


## 2. Full-release HTTP parity

Replay every one of the 136 stay-bearing users through the internal stay-event endpoint. This verifies API serialization/abstention behavior against direct production inference under CP2 v2.


In [ ]:
client=TestClient(app)
http_rows=[]

for user_id,user_stays in stays.groupby('user_id',sort=True):
    direct=infer_home_office(user_stays)
    payload={
        'user_id':str(user_id),
        'stays':[
            {
                'arrival_time_utc':pd.Timestamp(row.arrival_time_utc).isoformat(),
                'departure_time_utc':pd.Timestamp(row.departure_time_utc).isoformat(),
                'latitude':float(row.latitude),
                'longitude':float(row.longitude),
            }
            for row in user_stays.itertuples(index=False)
        ],
    }
    response=client.post('/v1/home-office/infer',json=payload)
    ok=response.status_code==200
    body=response.json() if ok else {}
    contract_ok=body.get('model_contract')=='cp2-v2' if ok else False
    http={item['label']:item for item in body.get('results',[]) if item.get('status')=='emitted'}
    direct_by={row.label:row for row in direct.itertuples(index=False)}
    mismatch=[]
    for label in ('HOME','OFFICE'):
        d=direct_by.get(label); h=http.get(label)
        if (d is None)!=(h is None): mismatch.append(label+':presence')
        elif d is not None:
            checks=[
                int(h['location_id'])==int(d.location_id),
                abs(float(h['evidence_strength'])-float(d.evidence_strength))<1e-9,
                abs(float(h['relevant_dwell_share'])-float(d.relevant_dwell_share))<1e-9,
                abs(float(h['share_margin'])-float(d.share_margin))<1e-9,
                int(h['relevant_dates'])==int(d.relevant_dates),
                abs(float(h['relevant_dwell_h'])-float(d.relevant_dwell_h))<1e-9,
            ]
            if not all(checks): mismatch.append(label+':evidence')
    http_rows.append({
        'user_id':str(user_id),'status_code':response.status_code,'model_contract_ok':contract_ok,
        'mismatch_count':len(mismatch),'mismatch':','.join(mismatch),
    })

http_parity=pd.DataFrame(http_rows)
display(http_parity[['status_code','model_contract_ok','mismatch_count']].agg(['count','min','max','sum']))
HTTP_PARITY_PASS=bool((http_parity['status_code']==200).all() and http_parity['model_contract_ok'].all() and (http_parity['mismatch_count']==0).all())
print('HTTP parity pass:',HTTP_PARITY_PASS)
if not HTTP_PARITY_PASS:
    display(http_parity.loc[http_parity['mismatch_count']>0].head(20))
    raise RuntimeError('full-release HTTP parity failed')


## 3. Freeze private CP2-v2 namespace artifacts

These artifacts are private and remain on the Modal Volume. They become downstream inputs only after the measured migration result is reviewed and the production PR is merged.


In [ ]:
semantic_stays,semantic_locations=build_semantic_locations(stays)
emissions=infer_home_office(stays)

semantic_stays.to_pickle(C/'semantic_stays_cp2_v2_private.pkl')
semantic_locations.to_pickle(C/'semantic_locations_cp2_v2_private.pkl')
emissions.to_pickle(C/'home_office_emissions_cp2_v2_private.pkl')
http_parity.to_csv(C/'http_parity_private.csv',index=False)
pd.DataFrame([comparison['decision']]).to_csv(C/'migration_decision.csv',index=False)

manifest={
    'model_contract':'cp2-v2',
    'location_namespace':comparison['location_namespace'],
    'git_sha':GIT_SHA,
    'stay_cache_path':str(STAY_CACHE),
    'stay_cache_sha256':STAY_CACHE_SHA256,
    'cp1_stays':int(len(stays)),
    'cp1_stay_users':int(stays['user_id'].nunique()),
    'timezone_resolved_stays':comparison['timezone_resolved_stays'],
    'timezone_resolved_users':comparison['timezone_resolved_users'],
    'distinct_timezones':comparison['distinct_timezones'],
    'semantic_locations':int(len(semantic_locations)),
    'recurring_locations':int((semantic_locations['stay_count']>=2).sum()),
    'recurring_users':int(semantic_locations.loc[semantic_locations['stay_count']>=2,'user_id'].nunique()),
    'home_emitted':int((emissions['label']=='HOME').sum()) if not emissions.empty else 0,
    'office_emitted':int((emissions['label']=='OFFICE').sum()) if not emissions.empty else 0,
    'unique_emitted_users':int(emissions['user_id'].nunique()) if not emissions.empty else 0,
    'migration_decision':comparison['decision'],
    'http_parity_pass':HTTP_PARITY_PASS,
    'historical_cp2_v1':comparison['historical_v1'],
}
(C/'cp2_v2_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')

print('saved:',C)
display(pd.Series(manifest).to_frame('value'))
print('RUNNER STATUS: ready_for_cp2_v2_refreeze_review')
